# M2 — Localização de Facilidades

**Treinamento de Otimização Aplicada — Genoa para Gradus**

## Caso: Beerlink expande — onde abrir os CDs?

Maurício cresceu. A Beerlink hoje atende **10 cidades** em Grande SP, ABC, Vale do Paraíba e interior. Está pagando frete avulso a CDs alugados mês a mês. Quer decidir, de forma estratégica, em quais cidades operar CDs próprios.

**5 sites candidatos** a CD. Cada um tem:
- Aluguel mensal (custo fixo de abrir)
- Capacidade em caixas/mês

**Frete:** R\$ 0,80 / cx / km, proporcional à distância CD → cidade.

Decisão dupla:
- $y_j \in \{0,1\}$: abrir o CD candidato $j$?
- $x_{ij} \ge 0$: fluxo (caixas/mês) do CD $j$ para a cidade $i$

É **Capacitated Facility Location Problem (CFLP)** — um dos MIPs clássicos.

## Setup

In [ ]:
%pip install -q ortools gurobipy

In [ ]:
import math
import pandas as pd

# 5 candidatos a CD
CANDIDATOS = ['SP-Pinheiros', 'Guarulhos', 'Campinas', 'Sorocaba', 'SJCampos']
ALUGUEL = {'SP-Pinheiros': 80, 'Guarulhos': 55, 'Campinas': 45, 'Sorocaba': 38, 'SJCampos': 42}  # R$ mil/mês
CAPAC   = {'SP-Pinheiros': 8000, 'Guarulhos': 6500, 'Campinas': 5500, 'Sorocaba': 4800, 'SJCampos': 5200}  # cx/mês

# 10 cidades atendidas
CIDADES = ['SP-Capital', 'Guarulhos', 'Osasco', 'ABC', 'Campinas',
           'Sorocaba', 'Jundiaí', 'SJCampos', 'Taubaté', 'Piracicaba']
DEMANDA = {'SP-Capital': 3500, 'Guarulhos': 1100, 'Osasco': 900, 'ABC': 1200, 'Campinas': 1500,
           'Sorocaba': 900, 'Jundiaí': 600, 'SJCampos': 1100, 'Taubaté': 600, 'Piracicaba': 500}

# Coordenadas aproximadas (lat, lon) — usadas só para calcular distâncias
COORDS = {
    'SP-Pinheiros': (-23.567, -46.685), 'Guarulhos': (-23.463, -46.533),
    'Campinas': (-22.907, -47.063),     'Sorocaba': (-23.501, -47.458),
    'SJCampos': (-23.179, -45.886),
    'SP-Capital': (-23.550, -46.633),   'Osasco': (-23.532, -46.792),
    'ABC': (-23.660, -46.561),          'Jundiaí': (-23.186, -46.884),
    'Taubaté': (-23.026, -45.555),      'Piracicaba': (-22.725, -47.649),
}

def km(a, b): return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2) * 111

TARIFA_KM = 0.80   # R$ / cx / km
FRETE = {(c, cd): km(COORDS[c], COORDS[cd]) * TARIFA_KM for c in CIDADES for cd in CANDIDATOS}

print(f'Demanda total: {sum(DEMANDA.values()):,} cx/mês')
print(f'Capacidade total (se todos os 5 CDs abrirem): {sum(CAPAC.values()):,} cx/mês')
print(f'Mín. {(sum(DEMANDA.values())+max(CAPAC.values())-1)//max(CAPAC.values())} CD(s) tecnicamente bastaria(m) — mas frete e dispersão geográfica forçam mais.')

## Modelo CFLP — formulação

$$\min \sum_j r_j y_j + \sum_{i,j} c_{ij} x_{ij}$$

Sujeito a:
- **Demanda:** $\sum_j x_{ij} = d_i \quad \forall i$
- **Capacidade (só se aberto):** $\sum_i x_{ij} \le Q_j\, y_j \quad \forall j$
- **Domínios:** $y_j \in \{0,1\},\ x_{ij} \ge 0$

O "truque" da multiplicação $Q_j \cdot y_j$ é o padrão clássico: se $y_j = 0$, o lado direito é zero (ninguém pode enviar nada). Se $y_j = 1$, vira a capacidade real.

### Solver 1 — OR-Tools (CBC)

In [ ]:
from ortools.linear_solver import pywraplp
import time

def solve_cflp_ortools():
    s = pywraplp.Solver.CreateSolver('CBC')
    y = {cd: s.IntVar(0, 1, f'y[{cd}]') for cd in CANDIDATOS}
    x = {(c, cd): s.NumVar(0, s.infinity(), f'x[{c},{cd}]')
         for c in CIDADES for cd in CANDIDATOS}
    # Demanda
    for c in CIDADES:
        s.Add(sum(x[c, cd] for cd in CANDIDATOS) == DEMANDA[c])
    # Capacidade
    for cd in CANDIDATOS:
        s.Add(sum(x[c, cd] for c in CIDADES) <= CAPAC[cd] * y[cd])
    # FO (R$ mil / mês)
    s.Minimize(
        sum(ALUGUEL[cd] * 1000 * y[cd] for cd in CANDIDATOS)
        + sum(FRETE[c, cd] * x[c, cd] for c in CIDADES for cd in CANDIDATOS)
    )
    t0 = time.time(); status = s.Solve(); t = time.time() - t0
    return {'custo': s.Objective().Value(),
            'tempo_ms': t * 1000,
            'abertos': {cd: y[cd].solution_value() > 0.5 for cd in CANDIDATOS},
            'fluxos': {(c, cd): x[c, cd].solution_value() for c in CIDADES for cd in CANDIDATOS}}

r_or = solve_cflp_ortools()
print(f"OR-Tools (CBC): custo R$ {r_or['custo']:,.0f}/mês  ({r_or['tempo_ms']:.0f} ms)")
for cd in CANDIDATOS:
    if r_or['abertos'][cd]:
        env = sum(r_or['fluxos'][c, cd] for c in CIDADES)
        print(f"  ✓ {cd:<14}: {env:>6,.0f} cx/mês  ({env/CAPAC[cd]*100:.0f}% util)")

### Solver 2 — Gurobi (gurobipy)

In [ ]:
import gurobipy as gp
from gurobipy import GRB

def solve_cflp_gurobi():
    m = gp.Model('cflp')
    m.Params.OutputFlag = 0
    y = m.addVars(CANDIDATOS, vtype=GRB.BINARY, name='y')
    x = m.addVars(CIDADES, CANDIDATOS, lb=0, name='x')
    m.addConstrs((gp.quicksum(x[c, cd] for cd in CANDIDATOS) == DEMANDA[c] for c in CIDADES), 'dem')
    m.addConstrs((gp.quicksum(x[c, cd] for c in CIDADES) <= CAPAC[cd] * y[cd] for cd in CANDIDATOS), 'cap')
    m.setObjective(
        1000 * gp.quicksum(ALUGUEL[cd] * y[cd] for cd in CANDIDATOS)
        + gp.quicksum(FRETE[c, cd] * x[c, cd] for c in CIDADES for cd in CANDIDATOS),
        GRB.MINIMIZE
    )
    t0 = time.time(); m.optimize(); t = time.time() - t0
    return {'custo': m.ObjVal, 'tempo_ms': t * 1000,
            'abertos': {cd: y[cd].X > 0.5 for cd in CANDIDATOS},
            'fluxos': {(c, cd): x[c, cd].X for c in CIDADES for cd in CANDIDATOS}}

r_gb = solve_cflp_gurobi()
print(f"Gurobi: custo R$ {r_gb['custo']:,.0f}/mês  ({r_gb['tempo_ms']:.0f} ms)")
for cd in CANDIDATOS:
    if r_gb['abertos'][cd]:
        env = sum(r_gb['fluxos'][c, cd] for c in CIDADES)
        print(f"  ✓ {cd:<14}: {env:>6,.0f} cx/mês  ({env/CAPAC[cd]*100:.0f}% util)")

### Lendo a solução

Os dois solvers convergem para o mesmo ótimo. **4 CDs abertos** (Guarulhos fica fora), custo total ≈ R\$ 330 mil/mês.

Por que Guarulhos não é aberto?
- SP-Pinheiros está perto e tem capacidade ampla (8000 cx)
- Pode atender SP-Capital + Guarulhos + Osasco + ABC numa só facility
- Abrir Guarulhos custaria R\$ 55k/mês de aluguel para economizar pouco frete

**Insight clássico de localização:** o solver consolida demanda em facilidades centrais (alta capacidade, próximas à maior demanda) mesmo que isso custe mais frete — porque o aluguel fixo de uma facility extra raramente compensa.

---

## Exercícios de extensão

### 1. Multi-produto (IPA + Pilsen com capacidades separadas)

Cada CD agora tem **capacidade dividida**: até 60 % para IPA, 40 % para Pilsen. Demanda também separada por produto.

Variáveis: $x_{ij}^{p}$ para cada produto $p$. Capacidade vira por produto: $\sum_i x_{ij}^p \le 0{,}6\, Q_j\, y_j$ (IPA) e $\le 0{,}4\, Q_j\, y_j$ (Pilsen).

In [ ]:
# Demanda separada por produto (60% IPA, 40% Pilsen — calibrado para que demanda total bata)
DEMANDA_PROD = {
    c: {'IPA': int(0.60 * DEMANDA[c]), 'Pilsen': DEMANDA[c] - int(0.60 * DEMANDA[c])}
    for c in CIDADES
}
PRODUTOS = ['IPA', 'Pilsen']
CAPAC_PROD = {
    (cd, 'IPA'):    int(0.60 * CAPAC[cd]),
    (cd, 'Pilsen'): CAPAC[cd] - int(0.60 * CAPAC[cd]),
}
CAPAC_PROD = {(cd, p): int((0.60 if p == 'IPA' else 0.40) * CAPAC[cd]) for cd in CANDIDATOS for p in PRODUTOS}

def solve_cflp_multiprod():
    m = gp.Model('cflp_mp'); m.Params.OutputFlag = 0
    y = m.addVars(CANDIDATOS, vtype=GRB.BINARY, name='y')
    x = m.addVars(CIDADES, CANDIDATOS, PRODUTOS, lb=0, name='x')
    m.addConstrs((gp.quicksum(x[c, cd, p] for cd in CANDIDATOS) == DEMANDA_PROD[c][p]
                  for c in CIDADES for p in PRODUTOS), 'dem')
    m.addConstrs((gp.quicksum(x[c, cd, p] for c in CIDADES) <= CAPAC_PROD[cd, p] * y[cd]
                  for cd in CANDIDATOS for p in PRODUTOS), 'cap')
    m.setObjective(
        1000 * gp.quicksum(ALUGUEL[cd] * y[cd] for cd in CANDIDATOS)
        + gp.quicksum(FRETE[c, cd] * x[c, cd, p] for c in CIDADES for cd in CANDIDATOS for p in PRODUTOS),
        GRB.MINIMIZE
    )
    m.optimize()
    return {'custo': m.ObjVal, 'abertos': {cd: y[cd].X > 0.5 for cd in CANDIDATOS}}

r_mp = solve_cflp_multiprod()
print(f"Multi-produto: custo R$ {r_mp['custo']:,.0f}/mês")
print(f"CDs abertos: {sum(r_mp['abertos'].values())}")
for cd, ab in r_mp['abertos'].items():
    if ab: print(f"  ✓ {cd}")

### 2. Redundância — cada cidade atendida por ≥ 2 CDs (resiliência)

Restrição extra: $\sum_j \mathbb{1}[x_{ij} > 0] \ge 2$ para cada cidade. Para linearizar, usamos uma variável auxiliar $z_{ij} \in \{0,1\}$ indicando se a cidade $i$ é atendida pelo CD $j$, ligada a $x_{ij}$ via big-M.

In [ ]:
def solve_cflp_redundancia():
    m = gp.Model('cflp_red'); m.Params.OutputFlag = 0
    y = m.addVars(CANDIDATOS, vtype=GRB.BINARY, name='y')
    x = m.addVars(CIDADES, CANDIDATOS, lb=0, name='x')
    z = m.addVars(CIDADES, CANDIDATOS, vtype=GRB.BINARY, name='z')
    BIG_M = max(DEMANDA.values())

    m.addConstrs((gp.quicksum(x[c, cd] for cd in CANDIDATOS) == DEMANDA[c] for c in CIDADES), 'dem')
    m.addConstrs((gp.quicksum(x[c, cd] for c in CIDADES) <= CAPAC[cd] * y[cd] for cd in CANDIDATOS), 'cap')
    # Redundância: cada cidade tem ≥ 2 CDs com fluxo > 0
    m.addConstrs((gp.quicksum(z[c, cd] for cd in CANDIDATOS) >= 2 for c in CIDADES), 'redund')
    # Acoplamento x -> z (se x > 0 então z = 1)
    m.addConstrs((x[c, cd] <= BIG_M * z[c, cd] for c in CIDADES for cd in CANDIDATOS), 'link')
    # z só se y aberto
    m.addConstrs((z[c, cd] <= y[cd] for c in CIDADES for cd in CANDIDATOS), 'open')

    m.setObjective(
        1000 * gp.quicksum(ALUGUEL[cd] * y[cd] for cd in CANDIDATOS)
        + gp.quicksum(FRETE[c, cd] * x[c, cd] for c in CIDADES for cd in CANDIDATOS),
        GRB.MINIMIZE
    )
    m.optimize()
    return {'custo': m.ObjVal, 'n_abertos': sum(int(y[cd].X > 0.5) for cd in CANDIDATOS)}

r_red = solve_cflp_redundancia()
print(f"Redundância (cada cidade atendida por ≥ 2 CDs): custo R$ {r_red['custo']:,.0f}/mês ({r_red['n_abertos']} CDs)")
print(f"Caso base (sem redundância):                       custo R$ {r_or['custo']:,.0f}/mês")
print(f"Custo da resiliência: R$ {r_red['custo'] - r_or['custo']:,.0f}/mês (+{(r_red['custo']/r_or['custo']-1)*100:.1f}%)")

### 3. p-median na escala — 20 cidades, 8 candidatos, exatamente p=3 CDs

Geramos uma instância maior e medimos tempo de OR-Tools vs Gurobi. Aqui o solver comercial costuma ganhar de fato:

In [ ]:
import random
random.seed(7)

# 8 candidatos sintéticos + 20 cidades em Grande SP
CAND20 = [f'CD{i+1:02d}' for i in range(8)]
CID20  = [f'Cid{i+1:02d}' for i in range(20)]
COORDS20 = {n: (-23.7 + random.random()*0.4, -47.0 + random.random()*0.7) for n in CAND20 + CID20}
DEMANDA20 = {c: random.randint(200, 1500) for c in CID20}
FRETE20 = {(c, cd): km(COORDS20[c], COORDS20[cd]) * TARIFA_KM for c in CID20 for cd in CAND20}
ALUGUEL20 = {cd: random.randint(30, 100) for cd in CAND20}    # R$ mil/mês
CAPAC20  = {cd: random.randint(4000, 9000) for cd in CAND20}
P = 3

def solve_pmedian(solver):
    if solver == 'gurobi':
        m = gp.Model(); m.Params.OutputFlag = 0
        y = m.addVars(CAND20, vtype=GRB.BINARY)
        x = m.addVars(CID20, CAND20, lb=0)
        m.addConstrs(gp.quicksum(x[c, cd] for cd in CAND20) == DEMANDA20[c] for c in CID20)
        m.addConstrs(gp.quicksum(x[c, cd] for c in CID20) <= CAPAC20[cd]*y[cd] for cd in CAND20)
        m.addConstr(gp.quicksum(y[cd] for cd in CAND20) == P)
        m.setObjective(1000*gp.quicksum(ALUGUEL20[cd]*y[cd] for cd in CAND20)
                       + gp.quicksum(FRETE20[c,cd]*x[c,cd] for c in CID20 for cd in CAND20), GRB.MINIMIZE)
        t0=time.time(); m.optimize(); return m.ObjVal, (time.time()-t0)*1000
    else:
        s = pywraplp.Solver.CreateSolver('CBC')
        y = {cd: s.IntVar(0,1,cd) for cd in CAND20}
        x = {(c,cd): s.NumVar(0, s.infinity(), f'x_{c}_{cd}') for c in CID20 for cd in CAND20}
        for c in CID20: s.Add(sum(x[c,cd] for cd in CAND20) == DEMANDA20[c])
        for cd in CAND20: s.Add(sum(x[c,cd] for c in CID20) <= CAPAC20[cd]*y[cd])
        s.Add(sum(y[cd] for cd in CAND20) == P)
        s.Minimize(sum(ALUGUEL20[cd]*1000*y[cd] for cd in CAND20)
                   + sum(FRETE20[c,cd]*x[c,cd] for c in CID20 for cd in CAND20))
        t0=time.time(); s.Solve(); return s.Objective().Value(), (time.time()-t0)*1000

c_or, t_or = solve_pmedian('ortools')
c_gb, t_gb = solve_pmedian('gurobi')
print(f'p-median 20 cidades × 8 candidatos × p={P}:')
print(f'  OR-Tools (CBC):  R$ {c_or:>10,.0f}/mês  ({t_or:>6.0f} ms)')
print(f'  Gurobi:          R$ {c_gb:>10,.0f}/mês  ({t_gb:>6.0f} ms)')
print(f'  Speed-up Gurobi: {t_or/t_gb:.1f}x mais rápido')

## Lições do M2

1. **CFLP é um MIP "limpo"** — binárias e contínuas, sem big-M no caso simples. Pareia bem com Gurobi.
2. **A multiplicação $Q_j \cdot y_j$** é o padrão para "ligar fluxo a binária de abrir". Memorize.
3. **Redundância é cara** — exigir 2 CDs por cidade aumenta significativamente o custo, mas é decisão estratégica (resiliência).
4. **Em escala** (20+ cidades, 10+ candidatos), o solver comercial Gurobi começa a fazer diferença mensurável. É o regime de problemas reais.

Conexão com o M1: lá decidimos rotas com CD fixo. Aqui decidimos o CD em si. Em consultoria, os dois problemas se combinam — "localização + roteamento" é o problema de design de rede de distribuição completo (e o terreno onde Gurobi + lazy constraints é estado da arte).